In [11]:


import re
import random
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
SEEDS = [42, 7, 21, 99, 123]
# ------------------------------------------------------------------
# PART 1 : SENTIMENT CLASSIFIER DATA
# ------------------------------------------------------------------
POS = ["wonderful", "amazing", "brilliant", "delightful", "excellent",
       "friendly", "kind", "great", "lovely", "fantastic"]
NEG = ["terrible", "awful", "horrible", "disgusting", "rude",
       "hateful", "dreadful", "nasty", "miserable", "worthless"]
# (marginalised term, majority term) pairs used for counterfactual swaps
PAIRS = [("muslim", "christian"), ("gay", "straight"), ("black", "white"),
         ("deaf", "hearing"), ("immigrant", "local"),
         ("transgender", "cisgender")]
ID_TEMPLATES = ["The {i} person was {w}.", "I think the {i} man is {w}.",
                "My {i} neighbour is really {w}.", "What a {w} {i} family!",
                "She is a {w} {i} woman.", "That {i} teacher seemed {w}."]
GEN_TEMPLATES = ["The movie was {w}.", "The service was {w}.",
                 "What a {w} day!", "The food tasted {w}.",
                 "The hotel staff were {w}.", "This phone is {w}."]
def make_sentiment(n, rng, biased, noise=0.08):
    """label 1 = negative (the 'flagged' class), 0 = positive."""
    X, y, grp = [], [], []
    for _ in range(n):
        has_id = rng.random() < 0.5
        marg = rng.random() < 0.5
        if has_id and biased:          # historical bias in collected data
            neg = rng.random() < (0.85 if marg else 0.15)
        else:
            neg = rng.random() < 0.5
        word_neg = neg if rng.random() > noise else (not neg)
        w = rng.choice(NEG if word_neg else POS)
        if has_id:
            pair = rng.choice(PAIRS)
            term = pair[0] if marg else pair[1]
            X.append(rng.choice(ID_TEMPLATES).format(i=term, w=w))
            grp.append(1 if marg else 0)
        else:
            X.append(rng.choice(GEN_TEMPLATES).format(w=w))
            grp.append(-1)
        y.append(int(neg))
    return X, np.array(y), np.array(grp)
def swap_identity(text):
    for a, b in PAIRS:
        if re.search(rf"\b{a}\b", text):
            return re.sub(rf"\b{a}\b", b, text)
        if re.search(rf"\b{b}\b", text):
            return re.sub(rf"\b{b}\b", a, text)
    return text
def mask_identity(text):
    terms = [t for p in PAIRS for t in p]
    return re.sub(r"\b(" + "|".join(terms) + r")\b", "idterm", text)
# ------------------------------------------------------------------
# PART 2 : RESUME SCREENER DATA
# ------------------------------------------------------------------
M_NAMES = ["Arjun", "Karthik", "Rahul", "Vikram", "Suresh", "Michael",
           "David", "John", "Rohan", "Aditya"]
F_NAMES = ["Priya", "Divya", "Ananya", "Lakshmi", "Meena", "Sarah",
           "Emily", "Sneha", "Kavya", "Anita"]
SURN = ["Kumar", "Raman", "Iyer", "Smith", "Reddy", "Nair", "Patel",
        "Johnson", "Menon", "Das"]
REL = ["python", "sql", "machine learning", "nlp", "statistics",
       "java", "cloud", "docker"]
IRR = ["ms office", "photoshop", "typing", "public speaking", "tally",
       "video editing", "drawing", "blogging"]
M_ACT = ["captain of the chess club", "member of the men's football team",
         "fraternity treasurer", "president of the engineering society",
         "head of the cricket club"]
F_ACT = ["captain of the women's chess club", "member of the women's football team",
         "sorority treasurer", "president of the women in engineering society",
         "head of the women's cricket club"]
SWAP = {"women's": "men's", "men's": "women's", "sorority": "fraternity",
        "fraternity": "sorority", "women in engineering society": "engineering society"}
def make_resumes(n, rng, bias_prob=0.35):
    """Returns text, historical label, merit label, gender (1 = female)."""
    texts, hist, merit, gen = [], [], [], []
    for _ in range(n):
        female = rng.random() < 0.5
        name = rng.choice(F_NAMES if female else M_NAMES) + " " + rng.choice(SURN)
        k_rel = rng.randint(0, 8)
        k_irr = rng.randint(0, 4)
        skills = rng.sample(REL, k_rel) + rng.sample(IRR, k_irr)
        rng.shuffle(skills)
        years = rng.randint(0, 10)
        score = 0.6 * k_rel / 8 + 0.4 * years / 10 + rng.gauss(0, 0.06)
        qualified = int(score > 0.5)
        h = qualified
        if female and qualified and rng.random() < bias_prob:
            h = 0                      # historical discrimination
        act = rng.choice(F_ACT if female else M_ACT)
        degree = rng.choice(["B.E. in CSE", "B.Tech in IT", "M.Sc in Data Science"])
        texts.append(f"{name}. {degree}. Skills: {', '.join(skills)}. "
                     f"Experience: {years} years exp{years}. Activities: {act}.")
        hist.append(h)
        merit.append(qualified)
        gen.append(int(female))
    return texts, np.array(hist), np.array(merit), np.array(gen)
def swap_gender(text):
    names = {**dict(zip(F_NAMES, M_NAMES)), **dict(zip(M_NAMES, F_NAMES))}
    first = text.split(" ")[0]
    text = text.replace(first, names.get(first, first), 1)
    for a, b in SWAP.items():
        if a in text:
            return text.replace(a, b)
    return text
def anonymise(text):
    text = re.sub(r"^[A-Za-z]+ [A-Za-z]+\.", "Candidate.", text)
    text = re.sub(r"sorority|fraternity", "house", text)
    text = re.sub(r"women in |women's |men's ", "", text)
    return text
# ------------------------------------------------------------------
# GENERIC HELPERS
# ------------------------------------------------------------------
def reweigh_weights(group, y):
    """Kamiran-Calders reweighing: w(g,y) = P(g)P(y) / P(g,y)."""
    w = np.ones(len(y))
    for g in np.unique(group):
        for c in np.unique(y):
            m = (group == g) & (y == c)
            if m.sum():
                w[m] = (np.mean(group == g) * np.mean(y == c)) / np.mean(m)
    return w
def fit(texts, y, weights=None):
    vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
    clf = LogisticRegression(max_iter=2000, C=3.0)
    clf.fit(vec.fit_transform(texts), y, sample_weight=weights)
    return vec, clf
def predict(model, texts, prep=None):
    vec, clf = model
    if prep:
        texts = [prep(t) for t in texts]
    return clf.predict(vec.transform(texts))
# ------------------------------------------------------------------
# EXPERIMENT 1 : SENTIMENT
# ------------------------------------------------------------------
def run_sentiment(seed):
    rng = random.Random(seed)
    Xtr, ytr, gtr = make_sentiment(6000, rng, biased=True)
    Xte, yte, gte = make_sentiment(3000, rng, biased=False)
    id_idx = np.where(gte >= 0)[0]
    out = {}
    def evaluate(name, model, prep=None, swap_fn=swap_identity):
        p = predict(model, Xte, prep)
        pos = (yte == 0)
        fpr = lambda g: np.mean(p[(gte == g) & pos] == 1)
        id_text = [Xte[i] for i in id_idx]
        flips = np.mean(predict(model, id_text, prep) !=
                        predict(model, [swap_fn(t) for t in id_text], prep))
        out[name] = dict(acc=accuracy_score(yte, p), f1=f1_score(yte, p),
                         fpr_marg=fpr(1), fpr_maj=fpr(0),
                         fpr_gap=fpr(1) - fpr(0), flip=flips)
    base = fit(Xtr, ytr)
    evaluate("Baseline (biased)", base)
    masked = [mask_identity(t) for t in Xtr]
    evaluate("Identity masking", fit(masked, ytr), prep=mask_identity)
    Xcda = Xtr + [swap_identity(t) for t in Xtr]
    evaluate("Counterfactual augmentation", fit(Xcda, np.concatenate([ytr, ytr])))
    m = gtr >= 0
    w = np.ones(len(ytr))
    w[m] = reweigh_weights(gtr[m], ytr[m])
    evaluate("Reweighing", fit(Xtr, ytr, w))
    return out
# ------------------------------------------------------------------
# EXPERIMENT 2 : RESUME SCREENER
# ------------------------------------------------------------------
def run_resume(seed):
    rng = random.Random(seed)
    Xtr, htr, mtr, gtr = make_resumes(6000, rng)
    Xte, hte, mte, gte = make_resumes(3000, rng)
    out = {}
    def evaluate(name, model, prep=None):
        p = predict(model, Xte, prep)
        sel = lambda g: np.mean(p[gte == g])
        tpr = lambda g: np.mean(p[(gte == g) & (mte == 1)])
        flips = np.mean(p != predict(model, [swap_gender(t) for t in Xte], prep))
        out[name] = dict(acc_merit=accuracy_score(mte, p),
                         acc_hist=accuracy_score(hte, p), f1=f1_score(mte, p),
                         sel_f=sel(1), sel_m=sel(0), di=sel(1) / sel(0),
                         eod=tpr(0) - tpr(1), flip=flips)
    evaluate("Baseline (biased)", fit(Xtr, htr))
    evaluate("Anonymisation", fit([anonymise(t) for t in Xtr], htr), prep=anonymise)
    Xcda = Xtr + [swap_gender(t) for t in Xtr]
    evaluate("Counterfactual augmentation", fit(Xcda, np.concatenate([htr, htr])))
    evaluate("Reweighing", fit(Xtr, htr, reweigh_weights(gtr, htr)))
    return out
def average(runs):
    names = runs[0].keys()
    return {n: {k: (np.mean([r[n][k] for r in runs]),
                    np.std([r[n][k] for r in runs])) for k in runs[0][n]}
            for n in names}
def show(title, res, keys):
    print("\n" + title)
    print("-" * 100)
    print(f"{'Model':<30}" + "".join(f"{k:>14}" for k in keys))
    for name, d in res.items():
        row = "".join(f"{d[k][0]:>14.3f}" if isinstance(d[k], tuple)
                      else f"{d[k]:>14.3f}" for k in keys)
        print(f"{name:<30}{row}")
def main():
    print("=== SINGLE RUN (seed = 42) ===")
    s1, r1 = run_sentiment(42), run_resume(42)
    show("CASE 1 - SENTIMENT CLASSIFIER", s1,
         ["acc", "f1", "fpr_marg", "fpr_maj", "fpr_gap", "flip"])
    show("CASE 2 - RESUME SCREENER", r1,
         ["acc_merit", "acc_hist", "f1", "sel_f", "sel_m", "di", "eod", "flip"])
    print("\n\n=== MEAN OVER 5 SEEDS", SEEDS, "===")
    sA = average([run_sentiment(s) for s in SEEDS])
    rA = average([run_resume(s) for s in SEEDS])
    show("CASE 1 - SENTIMENT CLASSIFIER (mean)", sA,
         ["acc", "f1", "fpr_marg", "fpr_maj", "fpr_gap", "flip"])
    show("CASE 2 - RESUME SCREENER (mean)", rA,
         ["acc_merit", "acc_hist", "f1", "sel_f", "sel_m", "di", "eod", "flip"])
    print("\nStd of accuracy across seeds:")
    for n in sA:
        print(f"  Sentiment {n:<30} {sA[n]['acc'][1]:.4f}")
    for n in rA:
        print(f"  Resume    {n:<30} {rA[n]['acc_merit'][1]:.4f}")
# Run the program (works in Google Colab / Jupyter and as a script)
main()

=== SINGLE RUN (seed = 42) ===

CASE 1 - SENTIMENT CLASSIFIER
----------------------------------------------------------------------------------------------------
Model                                    acc            f1      fpr_marg       fpr_maj       fpr_gap          flip
Baseline (biased)                      0.871         0.872         0.298         0.062         0.236         0.223
Identity masking                       0.920         0.921         0.090         0.095        -0.005         0.000
Counterfactual augmentation            0.920         0.921         0.090         0.095        -0.005         0.000
Reweighing                             0.919         0.920         0.093         0.090         0.003         0.011

CASE 2 - RESUME SCREENER
----------------------------------------------------------------------------------------------------
Model                              acc_merit      acc_hist            f1         sel_f         sel_m            di           eod       